## How would we execute a query like this?

``` sql
SELECT persons.*
FROM persons
    JOIN orders ON name = person_name
    JOIN books ON book_title = title
WHERE price < 10
```

In [1]:
import duckdb
import pandas as pd
import random
import queue

random.seed(21)

from faker import Faker



## Create fake data

In [2]:
from system.data_classes import Person, Book, Order

Faker.seed(15)
fake = Faker()

number_of_tuples = 3

# create some fake data mimicking tuples in a relation
persons = [Person(fake.name(), fake.date()) for _ in range(number_of_tuples)]
books = [
    Book(
        fake.sentence(),
        fake.name(),
        fake.pyfloat(min_value=0, max_value=100, right_digits=2),
    )
    for _ in range(number_of_tuples)
]
orders = []
i = 0
num_orders = 1.5 * number_of_tuples
assert number_of_tuples > 0
assert num_orders <= number_of_tuples * number_of_tuples
while i < num_orders:
    o = Order(
        persons[random.randint(0, number_of_tuples - 1)].name,
        books[random.randint(0, number_of_tuples - 1)].title,
    )
    if o in orders:
        continue
    orders.append(o)
    i += 1

display(persons, books, orders)

[Person(name='Alexandra Fitzgerald', birthday='1978-09-02'),
 Person(name='Mrs. Sherry Maldonado', birthday='2017-01-16'),
 Person(name='Mrs. Gina Erickson MD', birthday='1984-06-13')]

[Book(title='Various deal great finish figure.', author='Jennifer Ray', price=53.62),
 Book(title='Leave sometimes keep firm.', author='Joanne Frank', price=8.62),
 Book(title='Cultural move pass.', author='Sarah Smith', price=67.63)]

[Order(person_name='Alexandra Fitzgerald', book_title='Leave sometimes keep firm.'),
 Order(person_name='Mrs. Gina Erickson MD', book_title='Leave sometimes keep firm.'),
 Order(person_name='Mrs. Sherry Maldonado', book_title='Various deal great finish figure.'),
 Order(person_name='Mrs. Sherry Maldonado', book_title='Cultural move pass.'),
 Order(person_name='Alexandra Fitzgerald', book_title='Cultural move pass.')]

## Semi-join

In [3]:
from system.query_processing.operators import Relation, Filter, SHJ, SemiJ, Print

rel_persons = Relation("persons", [vars(e) for e in persons])
rel_orders = Relation("orders", [vars(e) for e in orders])
rel_books = Relation("books", [vars(e) for e in books])
filter_books = Filter(rel_books, "price < 10")
join_orders_books = SHJ(rel_orders, filter_books, "book_title", "title")
filtered_persons = SemiJ(rel_persons, join_orders_books, "name", "person_name")
query_plan = Print(filtered_persons)
query_plan.dump()

Print
  SemiJ on `name=person_name`
    Relation(persons)
    SHJ on `book_title=title`
      Relation(orders)
      Filter on `price < 10`
        Relation(books)


In [4]:
query_plan.interpret_open()

('Alexandra Fitzgerald', '1978-09-02')
('Mrs. Gina Erickson MD', '1984-06-13')


Since we're only interested in the qualifying tuples of a single relation, we can conveniently use a semi-join to filter the `persons` table based on the join result.
But how can we retrieve just the qualifying tuples from multiple relations *without* computing the full join result?

## Extend SQL: `SELECT RESULTDB`
Consider the database tables with sample data from above.

<div style="display: flex; gap: 20px;">

<!-- Customers Table -->
<table>
  <caption>persons</caption>
  <tr><th>name</th><th>birthday</th></tr>
  <tr><td>Alexandra Fitzgerald</td><td>1978-09-02</td></tr>
  <tr><td>Mrs. Sherry Maldonado</td><td>2017-01-14</td></tr>
  <tr><td>Mrs. Gina Erickson MD</td><td>1984-06-12</td></tr>
</table>

<!-- Orders Table -->
<table>
  <caption>orders</caption>
  <tr><th>person_name</th><th>book_title</th></tr>
  <tr><td>Alexandra Fitzgerald</td><td>Leave sometimes keep firm.</td></tr>
  <tr><td>Mrs. Gina Erickson MD</td><td>Leave sometimes keep firm.</td></tr>
  <tr><td>Mrs. Sherry Maldonado</td><td>Various deal great finish figure.</td></tr>
  <tr><td>Mrs. Sherry Maldonado</td><td>Cultural move pass.</td></tr>
  <tr><td>Alexandra Fitzgerald</td><td>Cultural move pass.</td></tr>
</table>

<!-- Books Table -->
<table>
  <caption>books</caption>
  <tr><th>title</th><th>author</th><th>price</th></tr>
  <tr><td>Various deal great finish figure.</td><td>Jennifer Ray</td><td>53.62</td></tr>
  <tr><td>Leave sometimes keep firm.</td><td>Joanne Frank</td><td>8.62</td></tr>
  <tr><td>Cultural move pass.</td><td>Sarah Smith</td><td>67.63</td></tr>
</table>

</div>


Executing the following query:
``` sql
SELECT RESULTDB *
FROM persons
    JOIN orders ON name = person_name
    JOIN books ON book_title = title
WHERE price < 10
```
would result in the following reduced relations:

<div style="display: flex; gap: 20px;">

<!-- Customers Table -->
<table>
  <caption>persons (reduced)</caption>
  <tr><th>name</th><th>birthday</th></tr>
  <tr><td>Alexandra Fitzgerald</td><td>1978-09-02</td></tr>
  <tr><td>Mrs. Gina Erickson MD</td><td>1984-06-12</td></tr>
</table>

<!-- Orders Table -->
<table>
  <caption>orders (reduced)</caption>
  <tr><th>person_name</th><th>book_title</th></tr>
  <tr><td>Alexandra Fitzgerald</td><td>Leave sometimes keep firm.</td></tr>
  <tr><td>Mrs. Sherry Maldonado</td><td>Leave sometimes keep firm.</td></tr>
</table>

<!-- Books Table -->
<table>
  <caption>books (reduced)</caption>
   <tr><th>title</th><th>author</th><th>price</th></tr>
  <tr><td>Leave sometimes keep firm.</td><td>Joanne Frank</td><td>8.62</td></tr>
</table>

</div>

In the following, we are going to discuss two possibilities to compute this result *subdatabase*.


## Rewrite Methods
In this section, we demonstrate two SQL-based rewrite methods that can be used to compute a result subdatabase.

Note: In the [paper](https://arxiv.org/pdf/2312.00638), we discuss two additional rewrite methods that use SQL subqueries to hint the query compiler to use semi-joins internally. However, this "[semi-join optimization](https://dev.mysql.com/doc/refman/8.0/en/semijoins.html)" depends on various factors and may not always be guaranteed.

In [5]:
# setup duckdb database
con = duckdb.connect(database=":memory:")

# convert to Pandas DataFrame
persons_df = pd.DataFrame(persons)
orders_df = pd.DataFrame(orders)
books_df = pd.DataFrame(books)

# import data
con.execute(f"CREATE TABLE persons AS SELECT * FROM persons_df")
con.execute(f"CREATE TABLE orders AS SELECT * FROM orders_df")
con.execute(f"CREATE TABLE books AS SELECT * FROM books_df")

### 1. Dynamic `SELECT DISTINCT`
This rewrite method transform a single `SELECT RESULTDB` query into multiple `SELECT DISTINCT` queries, one for each relation that is part of the query.

In [6]:
filtered_persons_dynamic = con.execute(
    """
    SELECT DISTINCT persons.*
    FROM persons
        JOIN orders ON name = person_name
        JOIN books ON book_title = title
    WHERE price < 10;
    """
)
display(filtered_persons_dynamic.fetch_df())
filtered_orders_dynamic = con.execute(
    """
    SELECT DISTINCT orders.*
    FROM persons
        JOIN orders ON name = person_name
        JOIN books ON book_title = title
    WHERE price < 10;
    """
)
display(filtered_orders_dynamic.fetch_df())
filtered_books_dynamic = con.execute(
    """
    SELECT DISTINCT books.*
    FROM persons
        JOIN orders ON name = person_name
        JOIN books ON book_title = title
    WHERE price < 10;
    """
)
display(filtered_books_dynamic.fetch_df())

,name,birthday
0,Alexandra Fitzgerald,1978-09-02
1,Mrs. Gina Erickson MD,1984-06-13


,person_name,book_title
0,Alexandra Fitzgerald,Leave sometimes keep firm.
1,Mrs. Gina Erickson MD,Leave sometimes keep firm.


,title,author,price
0,Leave sometimes keep firm.,Joanne Frank,8.62


This rewrite method has the drawback that essentially the same query is executed multiple times (except the `SELECT` clause).

### 2. Materialized `SELECT DISTINCT`
This rewrite method first materializes the necessary attributes of the single-table result in a temporary and unmaintained materialized view (MV) once.
Afterward, this MV is queried to obtain the individual result sets, and the MV is dropped again.

In [7]:
# duckdb does not support `CREATE MATERIALIZED VIEW`s, therefore simulate the MV
con.execute(
    """
    CREATE TABLE IF NOT EXISTS materialized_view AS
    SELECT *
    FROM persons
        JOIN orders ON name = person_name
        JOIN books ON book_title = title
    WHERE price < 10;
    """
)
filtered_persons_mat = con.execute(
    "SELECT DISTINCT name, birthday FROM materialized_view"
)
display(filtered_persons_mat.fetch_df())

filtered_orders_mat = con.execute(
    "SELECT DISTINCT person_name, book_title FROM materialized_view"
)
display(filtered_orders_mat.fetch_df())

filtered_books_mat = con.execute(
    "SELECT DISTINCT title, author, price FROM materialized_view"
)
display(filtered_books_mat.fetch_df())

,name,birthday
0,Alexandra Fitzgerald,1978-09-02
1,Mrs. Gina Erickson MD,1984-06-13


,person_name,book_title
0,Alexandra Fitzgerald,Leave sometimes keep firm.
1,Mrs. Gina Erickson MD,Leave sometimes keep firm.


,title,author,price
0,Leave sometimes keep firm.,Joanne Frank,8.62


This rewrite method has the advantage that it avoids the repeated computation of the same query.
However, the disadvantage is that we need to materialize a potentially large query result.
Overall, this rewrite method trades upfront costs for materializing the join result versus repeated cost for the computation of the join result.

## Native Algorithm to Compute a Result Subdatabase
In this section, we show how we can leverage semi-joins to implement a native $\text{ResultDB}_{\text{semi-join}}$ algorithm in order to compute a result subdatabase.

For this, we are going to make use of [Yannakakis' algorithm](https://www.researchgate.net/profile/Mihalis-Yannakakis/publication/200034379_Algorithms_for_Acyclic_Database_Schemes/links/5745c2a708ae9f741b430b62/Algorithms-for-Acyclic-Database-Schemes.pdf).
 Yannakakis' algorithm provides an efficient way for solving **acyclic** conjunctive queries, i.e., select-project-join queries with an acyclic join graph. The algorithm essentially works in three main steps.

```
1. Choose an arbitrary node in the join graph as root.
2. Perform bottom-up semi-joins from leaves to root. Perform top-down semi-joins from root to leaves.
3. Compute the final join result by joining the reduced relations.
```

In [8]:
def filter(relation: list[dict], pred: str) -> list[dict]:
    """Filter the given `relation` based on the predicate `pred`.
    :param relation: The relation to filter.
    :param pred: The predicate.
    :return: A list of tuples representing the filtered relation.
    """
    result: list[dict] = []
    for tup in relation:
        if eval(pred, dict(), tup):
            result.append(tup)
    return result


def semi_join(
    left_relation: list[dict],
    right_relation: list[dict],
    left_attribute: str,
    right_attribute: str,
) -> list[dict]:
    """Compute the left semi-join between the given relations, i.e., filter `left_relation` based on the join results with `right_relation`.
    :param left_relation: The relation to filter.
    :param right_relation: The relation on which `left_relation` gets filtered.
    :param left_attribute: The attribute of the left relation on which to filter.
    :param right_attribute: The attribute of the right relation on which to filter.
    :return: A list of tuples representing the filtered left relation.
    """
    result: list[dict] = []
    # build hashtable
    ht = set()
    for tup in right_relation:
        ht.add(tup[right_attribute])
    # probe hashtable
    for tup in left_relation:
        if tup[left_attribute] in ht:
            result.append(tup)
    return result

In [9]:
from system.query_optimization.join_graph import JoinGraph


def create_join_graph():
    """Create the join graph and necessary helper data structures for the running example.
    :return: The join graph, a mapping of relation id to relation name and tuples, and a mapping of relation ids to join attributes.
    """
    # create join graph
    join_graph = JoinGraph()

    # add relations
    persons_tuples = [vars(e) for e in persons]
    orders_tuples = [vars(e) for e in orders]
    books_tuples = [vars(e) for e in books]
    join_graph.add_relation(0)
    join_graph.add_relation(1)
    join_graph.add_relation(2)
    id_to_relation = {
        0: ("persons", persons_tuples),
        1: ("orders", orders_tuples),
        2: ("books", filter(books_tuples, "price < 10")),
    }

    # add joins
    join_graph.add_join(0, 1)
    join_graph.add_join(1, 2)
    ids_to_join = {
        (0, 1): ("name", "person_name"),
        (1, 0): ("person_name", "name"),
        (1, 2): ("book_title", "title"),
        (2, 1): ("title", "book_title"),
    }

    return join_graph, id_to_relation, ids_to_join

In [10]:
from system.query_optimization.subproblems import Subproblem


def compute_semi_join_order(join_graph: JoinGraph, root: int) -> list[tuple]:
    """Compute the semi join reduction order starting from the root.
    :param join_graph: The join graph representing the joins.
    :param root: The root node.
    :return: A list of tuples containing the relation ids in the order in which the semi-joins are to be executed.
    """
    # Compute BFS ordering starting at the root node
    semi_join_order: list[tuple] = []
    visited = set()
    Q = queue.Queue(len(join_graph))
    Q.put(root)

    while not Q.empty():
        x = Q.get()
        visited.add(x)
        neighbors = join_graph.get_neighbors(
            Subproblem.get_problem_for_relation(x)
        ).as_set()
        for n in neighbors:
            if n in visited:
                continue
            semi_join_order.append((x, n))
            Q.put(n)

    return semi_join_order


join_graph, _, _ = create_join_graph()
print(
    "Order of semi-joins with `persons` as root: ",
    compute_semi_join_order(join_graph, 0),
)
print(
    "Order of semi-joins with `orders` as root: ",
    compute_semi_join_order(join_graph, 1),
)
print(
    "Order of semi-joins with `books` as root: ", compute_semi_join_order(join_graph, 2)
)

Order of semi-joins with `persons` as root:  [(0, 1), (1, 2)]
Order of semi-joins with `orders` as root:  [(1, 0), (1, 2)]
Order of semi-joins with `books` as root:  [(2, 1), (1, 0)]


In [11]:
def semi_join_reduction(
    semi_join_order: list[tuple],
    id_to_relation: dict[int, tuple],
    ids_to_join: dict[tuple, tuple],
) -> None:
    """Perform the semi-joins based on the given order. The reduced relations are stored in `id_to_relation`.
    :param semi_join_order: The semi-join reduction order.
    :param id_to_relation: A dictionary mapping each relation id to a tuple containing the relation name and the corresponding tuples.
    :param ids_to_join: A dictionary mapping a tuple of relation ids representing a join to the corresponding attributes.
    """
    # bottom-up pass
    for left_rel_id, right_rel_id in reversed(semi_join_order):
        left_relation_name, left_relation_tuples = id_to_relation[left_rel_id]
        right_relation_name, right_relation_tuples = id_to_relation[right_rel_id]
        join_attributes = ids_to_join[(left_rel_id, right_rel_id)]
        semi_join_result = semi_join(
            left_relation_tuples, right_relation_tuples, *join_attributes
        )
        id_to_relation[left_rel_id] = (
            left_relation_name,
            semi_join_result,
        )  # replace left relation with filtered result

    # top-down pass
    for (
        right_rel_id,
        left_rel_id,
    ) in semi_join_order:  # switch left and right to ensure correct semi-join direction
        left_relation_name, left_relation_tuples = id_to_relation[left_rel_id]
        right_relation_name, right_relation_tuples = id_to_relation[right_rel_id]
        join_attributes = ids_to_join[(left_rel_id, right_rel_id)]
        semi_join_result = semi_join(
            left_relation_tuples, right_relation_tuples, *join_attributes
        )
        id_to_relation[left_rel_id] = (
            left_relation_name,
            semi_join_result,
        )  # replace left relation with filtered result


join_graph, id_to_relation, ids_to_join = create_join_graph()

print("State of relations before the semi-join reduction:")
display(id_to_relation)

print("State of relations after the semi-join reduction:")
semi_join_reduction(compute_semi_join_order(join_graph, 0), id_to_relation, ids_to_join)
display(id_to_relation)

State of relations before the semi-join reduction:


{0: ('persons',
  [{'name': 'Alexandra Fitzgerald', 'birthday': '1978-09-02'},
   {'name': 'Mrs. Sherry Maldonado', 'birthday': '2017-01-16'},
   {'name': 'Mrs. Gina Erickson MD', 'birthday': '1984-06-13'}]),
 1: ('orders',
  [{'person_name': 'Alexandra Fitzgerald',
    'book_title': 'Leave sometimes keep firm.'},
   {'person_name': 'Mrs. Gina Erickson MD',
    'book_title': 'Leave sometimes keep firm.'},
   {'person_name': 'Mrs. Sherry Maldonado',
    'book_title': 'Various deal great finish figure.'},
   {'person_name': 'Mrs. Sherry Maldonado',
    'book_title': 'Cultural move pass.'},
   {'person_name': 'Alexandra Fitzgerald',
    'book_title': 'Cultural move pass.'}]),
 2: ('books',
  [{'title': 'Leave sometimes keep firm.',
    'author': 'Joanne Frank',
    'price': 8.62}])}

State of relations after the semi-join reduction:


{0: ('persons',
  [{'name': 'Alexandra Fitzgerald', 'birthday': '1978-09-02'},
   {'name': 'Mrs. Gina Erickson MD', 'birthday': '1984-06-13'}]),
 1: ('orders',
  [{'person_name': 'Alexandra Fitzgerald',
    'book_title': 'Leave sometimes keep firm.'},
   {'person_name': 'Mrs. Gina Erickson MD',
    'book_title': 'Leave sometimes keep firm.'}]),
 2: ('books',
  [{'title': 'Leave sometimes keep firm.',
    'author': 'Joanne Frank',
    'price': 8.62}])}

In [12]:
from system.query_processing.operators import Collect


def yannakakis(
    join_graph: JoinGraph,
    id_to_relation: dict[int, tuple],
    ids_to_join: dict[tuple, tuple],
) -> list[tuple]:
    """Perform Yannakakis algorithm. The third and last step of this algorithm, as outlined above, joins the reduced relations to obtain the traditional result.
       Since we want to compute the result for our example query, we specifically construct the corresponding query execution plan in step three.
       Note, that normally the query optimizer would decide on the best join order which would then be executed by the query execution engine.
    :param join_graph: The join graph.
    :param id_to_relation: A dictionary mapping each relation id to a tuple containing the relation name and the corresponding tuples.
    :param ids_to_join: A dictionary mapping a tuple of relation ids representing a join to the corresponding attributes.
    """
    # 1. Choose arbitrary root node
    random.seed(42)
    root = random.choice(list(id_to_relation.keys()))

    # 2. Perform semi-joins
    semi_join_order = compute_semi_join_order(join_graph, root)
    semi_join_reduction(semi_join_order, id_to_relation, ids_to_join)

    # 3. Perform joins to obtain a
    persons_reduced = Relation(*id_to_relation[0])
    orders_reduced = Relation(*id_to_relation[1])
    books_reduced = Relation(
        *id_to_relation[2]
    )  # note that the filter on books was already applied
    join_orders_books = SHJ(orders_reduced, books_reduced, *ids_to_join[(1, 2)])
    join_persons_orders_books = SHJ(
        persons_reduced, join_orders_books, *ids_to_join[(0, 1)]
    )
    query_plan = Collect(join_persons_orders_books)
    query_plan.interpret_open()

    return query_plan.result


join_graph, id_to_relation, ids_to_join = create_join_graph()
result = yannakakis(join_graph, id_to_relation, ids_to_join)
display(result)

[('Leave sometimes keep firm.',
  'Joanne Frank',
  8.62,
  'Alexandra Fitzgerald',
  'Leave sometimes keep firm.',
  'Alexandra Fitzgerald',
  '1978-09-02'),
 ('Leave sometimes keep firm.',
  'Joanne Frank',
  8.62,
  'Mrs. Gina Erickson MD',
  'Leave sometimes keep firm.',
  'Mrs. Gina Erickson MD',
  '1984-06-13')]

In [13]:
# The implementation just uses the first two steps of the Yannakakis algorithm
def resultDB(
    join_graph: JoinGraph,
    id_to_relation: dict[int, tuple],
    ids_to_join: dict[tuple, tuple],
) -> dict[int, tuple]:
    """Compute a result subdatabase.
    :param join_graph: The join graph.
    :param id_to_relation: A dictionary mapping each relation id to a tuple containing the relation name and the corresponding tuples.
    :param ids_to_join: A dictionary mapping a tuple of relation ids representing a join to the corresponding attributes.
    :return: A dictionary mapping each relation id to a tuple containing the relation name and the reduced tuples.
    """
    # 1. Choose arbitrary root node
    random.seed(42)
    root = random.choice(list(id_to_relation.keys()))

    # 2. Perform semi-joins
    semi_join_order = compute_semi_join_order(join_graph, root)
    semi_join_reduction(semi_join_order, id_to_relation, ids_to_join)

    return id_to_relation


join_graph, id_to_relation, ids_to_join = create_join_graph()
result = resultDB(join_graph, id_to_relation, ids_to_join)
display(result)

{0: ('persons',
  [{'name': 'Alexandra Fitzgerald', 'birthday': '1978-09-02'},
   {'name': 'Mrs. Gina Erickson MD', 'birthday': '1984-06-13'}]),
 1: ('orders',
  [{'person_name': 'Alexandra Fitzgerald',
    'book_title': 'Leave sometimes keep firm.'},
   {'person_name': 'Mrs. Gina Erickson MD',
    'book_title': 'Leave sometimes keep firm.'}]),
 2: ('books',
  [{'title': 'Leave sometimes keep firm.',
    'author': 'Joanne Frank',
    'price': 8.62}])}

## Unit Tests

In [14]:
import unittest


class ResultDBTest(unittest.TestCase):
    @staticmethod
    def create_data():
        persons = [
            Person("John", "1992-17-03"),
            Person("Bob", "1987-08-05"),
            Person("Jane", "1968-12-12"),
        ]
        orders = [
            Order("John", "Echoes of Time"),
            Order("Jane", "Echoes of Time"),
            Order("Bob", "Shadows of Dawn"),
            Order("Bob", "Echoes of Time"),
            Order("John", "Shadows of Dawn"),
            Order("Jane", "Beneath the Stars"),
        ]
        books = [
            Book("Forgotten Dreams", "Alice", 54.92),
            Book("Echoes of Time", "Marie", 44.09),
            Book("Shadows of Dawn", "Alan", 32.99),
            Book("Beneath the Stars", "Ted", 78.21),
        ]
        return (
            [vars(e) for e in persons],
            [vars(e) for e in orders],
            [vars(e) for e in books],
        )

    def test_filter(self):
        persons, _, _ = self.create_data()
        filtered_persons = filter(persons, "birthday < '1990-08-12'")
        result = [
            {"name": "Bob", "birthday": "1987-08-05"},
            {"name": "Jane", "birthday": "1968-12-12"},
        ]
        self.assertListEqual(filtered_persons, result)

    def test_semi_join(self):
        _, orders, books = self.create_data()
        semi_join_result = semi_join(books, orders, "title", "book_title")
        result = [
            {"title": "Echoes of Time", "author": "Marie", "price": 44.09},
            {"title": "Shadows of Dawn", "author": "Alan", "price": 32.99},
            {"title": "Beneath the Stars", "author": "Ted", "price": 78.21},
        ]
        self.assertListEqual(semi_join_result, result)

    def test_compute_semi_join_order(self):
        join_graph = JoinGraph()
        join_graph.add_relation(0)
        join_graph.add_relation(1)
        join_graph.add_relation(2)
        join_graph.add_relation(3)
        join_graph.add_join(0, 1)
        join_graph.add_join(0, 2)
        join_graph.add_join(0, 3)
        semi_join_order = compute_semi_join_order(join_graph, 3)
        result = [(3, 0), (0, 1), (0, 2)]
        self.assertListEqual(semi_join_order, result)

    def test_semi_join_reduction(self):
        persons, orders, books = self.create_data()

        # create join graph
        join_graph = JoinGraph()
        join_graph.add_relation(0)
        join_graph.add_relation(1)
        join_graph.add_relation(2)
        id_to_relation = {
            0: ("persons", persons),
            1: ("orders", orders),
            2: ("books", filter(books, "price < 50")),
        }

        join_graph.add_join(0, 1)
        join_graph.add_join(1, 2)
        ids_to_join = {
            (0, 1): ("name", "person_name"),
            (1, 0): ("person_name", "name"),
            (1, 2): ("book_title", "title"),
            (2, 1): ("title", "book_title"),
        }

        # compute semi-join order
        semi_join_order = compute_semi_join_order(join_graph, 0)

        # execute semi-joins
        semi_join_reduction(semi_join_order, id_to_relation, ids_to_join)

        persons_expected = [
            {"name": "John", "birthday": "1992-17-03"},
            {"name": "Bob", "birthday": "1987-08-05"},
            {"name": "Jane", "birthday": "1968-12-12"},
        ]
        orders_expected = [
            {"person_name": "John", "book_title": "Echoes of Time"},
            {"person_name": "Jane", "book_title": "Echoes of Time"},
            {"person_name": "Bob", "book_title": "Shadows of Dawn"},
            {"person_name": "Bob", "book_title": "Echoes of Time"},
            {"person_name": "John", "book_title": "Shadows of Dawn"},
        ]
        books_expected = [
            {"title": "Echoes of Time", "author": "Marie", "price": 44.09},
            {"title": "Shadows of Dawn", "author": "Alan", "price": 32.99},
        ]
        persons_result = id_to_relation[0][1]
        orders_result = id_to_relation[1][1]
        books_result = id_to_relation[2][1]

        self.assertListEqual(persons_expected, persons_result)
        self.assertListEqual(orders_expected, orders_result)
        self.assertListEqual(books_expected, books_result)


# Run unit tests
unittest.main(argv=["ignored", "-v"], verbosity=2, exit=False)

test_compute_semi_join_order (__main__.ResultDBTest.test_compute_semi_join_order) ... ok
test_filter (__main__.ResultDBTest.test_filter) ... ok
test_semi_join (__main__.ResultDBTest.test_semi_join) ... ok
test_semi_join_reduction (__main__.ResultDBTest.test_semi_join_reduction) ... ok

----------------------------------------------------------------------
Ran 4 tests in 0.002s

OK
